# 04 · Confronto: engine × agente revisor (LLM)

**Pergunta:** um LLM revisando a saída da engine agrega algo, e dá para confiar nele?

**O agente** (`src/klike/agent.py`) recebe o JSON da engine para uma campanha e pode consultar os dados com 3 ferramentas só de leitura (`src/klike/agent_tools.py`): perfil da campanha contra a mediana da plataforma, pontos por atributo do modelo A e lift de um atributo em qualquer métrica. Ele devolve uma ordem sugerida, um comentário e uma execução criativa por ação, alertas e um resumo. **Ele não calcula nada**: a resposta passa por guardrails em código (ações idênticas às da engine, classificação de evidência copiada, hipóteses no fim, números rastreáveis até a engine ou as ferramentas, heurística contra promessas sem evidência). Se falhar, recebe as violações e tem uma chance de corrigir; se falhar de novo, é descartada e vale só a engine.

**Amostra:** 18 campanhas, 6 por plataforma (sorteio com `random_state=0` entre as que têm ao menos 3 recomendações). Modelo `gemini-3.5-flash-lite`, temperatura 0. As revisões foram geradas por `python scripts/run_agent_comparison.py` e estão salvas em `reports/agent/cache/`: este notebook roda sem chave de API.

**O que fica de fora:** a qualidade da execução criativa ("esse hook é bom?") não é medida aqui; isso exigiria avaliação humana ou um teste A/B.

In [1]:
import pandas as pd
from klike.agent import format_review
from klike.agent_eval import load_reviews, sample_campaigns, summary_table, tool_usage, violations_table
from klike.data import load_clean

pd.set_option("display.max_colwidth", 140)
df = load_clean()
ids = sample_campaigns(df)
v4, v5 = load_reviews(df, ids, version="v4"), load_reviews(df, ids)
print(f"{len(ids)} campanhas · revisões v4: {len(v4)} · v5: {len(v5)}")

18 campanhas · revisões v4: 18 · v5: 18


## 1. Confiabilidade: os guardrails e uma iteração do prompt

A primeira rodada (v4) teve **5 de 18 revisões rejeitadas**. Olhando as violações, o problema era metade meu e metade do formato:

- **Falso positivo do guardrail:** a regra "nenhum número na execução criativa" reprovava textos legítimos como "proporção 1:1" ou "entre 11 e 15 segundos", que vêm da própria ação. A regra virou "sem % nem R$, e todo número rastreável", igual ao resto do texto.
- **Formato:** respostas vazias (o loop de ferramentas acabava antes da resposta), JSON cortado e campos com nome errado. Na v5, o agente tem duas fases: **investigação** (com ferramentas) e **resposta** (sem ferramentas, com o schema JSON imposto pela API).

A heurística de "promessa sem evidência" pegou casos reais nas duas versões (ex.: "ideal para elevar o engajamento" numa ação cuja evidência de engajamento não é clara).

In [2]:
def reliability(reviews):
    t = summary_table(reviews)
    return pd.Series({
        "revisões": len(t),
        "aprovadas": int(t["aprovado"].sum()),
        "aprovadas de primeira": int((t["aprovado"] & (t["correções"] == 0)).sum()),
        "rejeitadas": int((~t["aprovado"]).sum()),
    })

pd.DataFrame({"v4": reliability(v4), "v5": reliability(v5)})

,v4,v5
revisões,18,18
aprovadas,13,17
aprovadas de primeira,8,15
rejeitadas,5,1


In [3]:
viol = pd.concat([violations_table(v4).assign(versão="v4"), violations_table(v5).assign(versão="v5")])
viol.pivot_table(index="tipo", columns=["versão", "tentativa"], values="campanha", aggfunc="count", fill_value=0)

versão                      v4    v5   
tentativa                    1  2  1  2
tipo                                   
JSON inválido                5  0  0  0
estrutura (ações/ordem)      0  2  1  0
número na execução criativa  5  5  0  0
número não rastreável        0  0  2  1
promessa sem evidência       1  1  0  0

## 2. Concordância: o agente muda a prioridade da engine?

A engine ordena por ganho no score ÷ esforço. O agente pode manter ou reordenar. Kendall τ = 1 é a mesma ordem; 0 é nenhuma relação; negativo é ordem invertida.

In [4]:
t5 = summary_table(v5)
ok = t5[t5["aprovado"]]
pd.Series({
    "revisões aprovadas": len(ok),
    "mesma ação em 1º": f"{ok['mesmo top1'].mean():.0%}",
    "mesma ordem completa": f"{ok['mesma ordem'].mean():.0%}",
    "Kendall τ médio": round(ok["kendall τ"].mean(), 2),
})

revisões aprovadas       17
mesma ação em 1º        65%
mesma ordem completa    65%
Kendall τ médio         0.8
dtype: object

In [5]:
ok[["plataforma", "objetivo", "top1 engine", "evid. top1 engine", "top1 agente", "evid. top1 agente", "kendall τ"]]

,plataforma,objetivo,top1 engine,evid. top1 engine,top1 agente,evid. top1 agente,kendall τ
campanha,,,,,,,
KLK-0042,Meta,app_install,hook,forte,hook,forte,1.00
KLK-0110,Meta,engagement,text,nao_clara,face,forte,0.67
KLK-0084,Meta,conversions,hook,forte,hook,forte,1.00
KLK-0096,Meta,conversions,text,nao_clara,text,nao_clara,1.00
KLK-0284,Meta,conversions,hook,forte,hook,forte,1.00
KLK-0226,Meta,conversions,cta,nao_clara,cta,nao_clara,1.00
KLK-0048,TikTok,awareness,text,nao_clara,hook,nao_clara,0.80
KLK-0220,TikTok,awareness,subtitle,nao_clara,hook,nao_clara,0.80
KLK-0200,TikTok,traffic,hook,forte,hook,forte,1.00


## 3. Quando discorda, por quê?

Hipótese: a engine pode pôr em 1º uma ação barata com evidência fraca (ganho ÷ esforço), e o agente prefere a ação com evidência forte na métrica de negócio. A tabela cruza a evidência da ação que cada um põe em 1º.

In [6]:
pd.DataFrame({
    "engine": ok["evid. top1 engine"].value_counts(),
    "agente": ok["evid. top1 agente"].value_counts(),
}).fillna(0).astype(int)

,engine,agente
forte,7,11
nao_clara,10,6


In [7]:
changed = [cid for cid in ok.index if not ok.loc[cid, "mesmo top1"]]
pd.DataFrame({cid: {"engine → agente": f"{ok.loc[cid, 'top1 engine']} → {ok.loc[cid, 'top1 agente']}",
                    "justificativa": v5[cid]["revisao"]["justificativa_ordem"]} for cid in changed}).T

,engine → agente,justificativa
KLK-0110,text → face,"A ordem foi alterada para priorizar a ação com evidência forte primeiro, já que a campanha possui engajamento abaixo da mediana da plata..."
KLK-0048,text → hook,"Reordenamos a ação de hook para o primeiro lugar por causa do ganho potencial de 16,9 pts e porque o objetivo de awareness no TikTok se ..."
KLK-0220,subtitle → hook,"A ordem foi ajustada priorizando o hook por ter demonstrado aumento no watch_ratio e possuir o maior ganho de score, seguido pelas demai..."
KLK-0114,cta → face,"Reordenamos para colocar a ação com evidência forte no topo, pois o ganho de 12,4 pts é o maior e mais seguro. A ação de duration ficou ..."
KLK-0314,text → subtitle,"A ordem foi reoordenada priorizando a ação com evidência forte para o objetivo de engajamento no LinkedIn, colocando as de evidência não..."
KLK-0337,text → subtitle,"A acao de adicionar legendas possui evidência forte e menor esforço, devendo liderar a prioridade para elevar o score e o engajamento."


## 4. O que o agente traz que a engine não traz

A engine só olha uma métrica (CTR ou engajamento, conforme o objetivo) e não olha a performance da própria campanha. O agente consulta o perfil da campanha e o efeito dos atributos em outras métricas (conversão, retenção).

In [8]:
usage = tool_usage(v5)
usage.fillna({"métrica": "—"}).groupby(["ferramenta", "métrica"]).size().rename("chamadas").to_frame()

chamadas
ferramenta      métrica                  
lift_metrica    conversion_rate        26
                ctr                     7
                engagement_rate         3
                roas                    1
                watch_ratio            18
perfil_campanha —                      18

In [9]:
alerts = [(cid, a) for cid in ok.index for a in v5[cid]["revisao"]["alertas"]]
print(f"{len(alerts)} alertas em {len(ok)} revisões ({len(alerts) / len(ok):.1f} por campanha). Exemplos:")
for cid, a in alerts[:6]:
    print(f"- {cid}: {a}")

33 alertas em 17 revisões (1.9 por campanha). Exemplos:
- KLK-0042: O CTR real de 0,0352 está consideravelmente abaixo da mediana da plataforma de 0,0646.
- KLK-0042: O klike_score real de 51,0 está abaixo da mediana da plataforma de 60,5.
- KLK-0110: A taxa de engajamento atual de 0,0067 está significativamente abaixo da mediana da plataforma de 0,018.
- KLK-0110: O CTR de 0,0285 também se encontra abaixo da mediana da plataforma de 0,0646.
- KLK-0084: O CTR real esta em 0,0259 enquanto a mediana da plataforma e 0,0646.
- KLK-0084: O klike_score real esta em 56,3 pontos, abaixo da previsao de 60,0.


## 5. Custo

In [10]:
pd.Series({
    "tokens por revisão (mediana)": int(t5["tokens"].median()),
    "tokens no total (18 revisões)": int(t5["tokens"].sum()),
    "consultas a ferramentas por revisão (média)": round(t5["consultas"].mean(), 1),
    "latência por revisão, s (mediana)": t5["latência (s)"].median(),
})

tokens por revisão (mediana)                    13526.00
tokens no total (18 revisões)                  267596.00
consultas a ferramentas por revisão (média)         4.10
latência por revisão, s (mediana)                  49.75
dtype: float64

A latência inclui as esperas por limite de requisições da camada gratuita da API; numa conta paga, cada revisão leva poucos segundos. A engine sozinha responde em milissegundos.

## 6. Exemplos lado a lado

In [11]:
example_changed = changed[0] if changed else ok.index[0]
print(format_review(v5[example_changed]))

Revisão do agente · KLK-0110 · Meta · engagement
Ordem da engine:  text → face → format_quadrado → duration
Ordem do agente:  face → text → format_quadrado → duration  (reordenada)
Por quê: A ordem foi alterada para priorizar a ação com evidência forte primeiro, já que a campanha possui engajamento abaixo da mediana da plataforma. A ação baseada em hipótese foi mantida obrigatoriamente no final.

1. Mostrar um rosto humano [evidência: forte]
   Mostrar um rosto humano é a ação de maior impacto, gerando 11,2 pts no score. A análise de watch_ratio confirmou que a presença de rostos aumenta a retenção no Meta.
   Execução: Insira uma pessoa da faixa etária de 45+ anos sorrindo e apresentando o [produto] logo nos primeiros segundos.
2. Aumentar um pouco o texto na tela (densidade média) [evidência: nao_clara]
   Aumentar a densidade do texto traz um ganho de 7,4 pts no score. O lift em watch_ratio mostrou que a diferença não é clara, indicando que o impacto precisa de teste para esta campa

In [12]:
example_kept = next(cid for cid in ok.index if ok.loc[cid, "mesma ordem"])
print(format_review(v5[example_kept]))

Revisão do agente · KLK-0042 · Meta · app_install
Ordem da engine:  hook → cta → face → format_quadrado → duration
Ordem do agente:  hook → cta → face → format_quadrado → duration
Por quê: A ordem da engine foi mantida porque coloca a ação de evidência forte no topo, seguida pelas de evidência nao_clara com bom ganho em pontos e esforço compatível, deixando a hipótese por último conforme as regras.

1. Adicionar um hook nos 3 primeiros segundos [evidência: forte]
   Esta ação traz o maior ganho de 16,1 pts no score, o que é essencial para elevar a campanha acima da mediana da plataforma. A análise de lift mostrou que o hook aumenta o watch_ratio no Meta.
   Execução: Corte direto para a tela do app [produto] mostrando a notificação de ganho em 3 segundos.
2. Incluir um CTA explícito [evidência: nao_clara]
   O ganho previsto é de 8,0 pts e o esforço é baixo, embora a evidência seja nao_clara para a conversão.
   Execução: Insira um botão fixo no final com o texto Baixe agora o [produto

## 7. Conclusões

1. **Com guardrails em código, o agente é confiável o bastante para uma camada opcional.** Da v4 para a v5, as revisões aprovadas nas validações foram de 13 para 17 de 18 (15 de primeira). Ser aprovada não exige concordar com a engine; a concordância é medida no item 2. A única rejeição da v5 foi um falso positivo ("proporção 16:9" lida como número inventado), corrigido no código depois da rodada (proporções de tela ficam fora da checagem). Não rodei a amostra de novo de propósito: continuar ajustando regras em cima das mesmas 18 campanhas seria sobreajuste. Nenhum número inventado passou, e as afirmações que conferi à mão batem com as ferramentas.
2. **O agente concorda quando a evidência é forte e discorda quando é fraca.** Quando a engine põe em 1º uma ação com evidência forte, o agente manteve em 7 de 7 casos. Quando ela põe em 1º uma ação de evidência não clara (o "ganho ÷ esforço" favorece ações baratas), o agente trocou em 6 de 10: em 4 por uma ação com evidência forte, em 2 pelo hook, apoiado em outra métrica.
3. **Ele achou um ponto fraco da engine.** Nas campanhas de awareness do TikTok, a engine mede a evidência pelo engajamento, em que o hook não tem efeito claro; o agente consultou a retenção (watch ratio), métrica mais próxima de awareness, e o hook aumenta a retenção em +34% (IC 90%: +22% a +63%). Isso sugere duas melhorias **determinísticas** na engine, sem LLM: usar a retenção como métrica de awareness e usar a força da evidência como critério na prioridade.
4. **Ele traz contexto da campanha que a engine não usa.** Consultou o perfil em 18 de 18 casos e gerou ~2 alertas por campanha (ex.: engajamento muito abaixo da mediana da plataforma), além de uma execução criativa concreta por ação.
5. **Custo:** ~13,5 mil tokens por revisão (mediana) e ~268 mil nas 18. A latência mediana de ~50 s é dominada pelas esperas de limite da camada gratuita.

**Leitura final:** a engine continua sendo o núcleo (rápida, determinística, auditável). O agente vale como revisor opcional, e o achado mais útil dele é o item 3: ele funcionou como **auditor da engine**, apontando melhorias que depois podem virar regra sem precisar do LLM.
